# Deepen the essentiality MSA with 8,388 bacterial species

Anchor-search our ~330k labeled-organism proteins against 8,388 dereplicated RefSeq complete bacterial genomes (one per species) to deepen the unsupervised evolutionary features, then retrain and measure the gain.

**Why anchor-search, not eggNOG clustering**: we only need orthologs of the genes we already have labels for — ~330k queries vs ~25M targets is a few-hour DIAMOND job, not a multi-day all-vs-all clustering.

**What it deepens** (unsupervised only — no new labels):
- `deep_retention`: # of the 8,388 species carrying a homolog (phyletic breadth, 48 -> 8,388 organisms)
- `deep_pident`: mean homolog identity (a cheap selection proxy)
- (phase 2, optional) deep dN/dS and phyletic-profile coevolution

**Honest ceiling**: adds zero essentiality labels, so the supervised ortholog-vote stays at the labeled organisms and the rogue/conditional zone stays shut. This sharpens the conserved-core prediction; it is the AlphaFold-depth move on the feature side.

**Disk**: ~12 GB proteins download + ~6 GB DIAMOND DB + working set ~ 40 GB peak. Fits your 200 GB.

## 1. Install tools + clone repo

In [ ]:
!pip install -q ncbi-datasets-cli
# DIAMOND binary (fast protein aligner)
!wget -q https://github.com/bbuchfink/diamond/releases/download/v2.1.9/diamond-linux64.tar.gz
!tar xzf diamond-linux64.tar.gz && chmod +x diamond && ./diamond version
!git clone --depth 1 -b claude/vectorize-gex-propensity-NRqBW https://github.com/nikku03/cell.git cell_repo || echo cloned
import os; os.chdir('cell_repo'); print('cwd', os.getcwd())

## 2. Build the 8,388 dereplicated accession list

In [ ]:
import json, collections, subprocess
subprocess.run('datasets summary genome taxon bacteria --assembly-level complete '
               '--assembly-source RefSeq > /tmp/summary.json', shell=True, check=True)
d = json.load(open('/tmp/summary.json'))
def species(r):
    p = r['organism']['organism_name'].split(); return ' '.join(p[:2]) if len(p)>=2 else p[0]
def score(r):
    cat = r.get('assembly_info',{}).get('refseq_category','na')
    cr = {'reference genome':2,'representative genome':1}.get(cat,0)
    ln = int(r.get('assembly_stats',{}).get('total_sequence_length',0) or 0)
    return (cr, ln)
best={}
for r in d['reports']:
    s=species(r)
    if s not in best or score(r)>score(best[s]): best[s]=r
accs=[best[s]['accession'] for s in best]
open('/tmp/derep.txt','w').write('\n'.join(accs)+'\n')
print('dereplicated species reps:', len(accs))

## 3. Download proteins for the 8,388 species (~12 GB) and concatenate

In [ ]:
# download proteins only (dehydrated -> rehydrate is robust for big sets)
!datasets download genome accession --inputfile /tmp/derep.txt \
    --include protein --dehydrated --filename /tmp/prot.zip
!unzip -o -q /tmp/prot.zip -d /tmp/protpkg
!datasets rehydrate --directory /tmp/protpkg --max-workers 12
# concatenate all protein.faa into one DB fasta, tagging each header with its genome accession
import glob, os
out=open('/tmp/db.faa','w'); n_p=n_g=0
for fp in glob.glob('/tmp/protpkg/ncbi_dataset/data/GCF_*/protein.faa'):
    acc=fp.split('/')[-2]; n_g+=1
    for line in open(fp):
        if line.startswith('>'):
            out.write('>'+acc+'|'+line[1:].split()[0]+'\n'); n_p+=1
        else: out.write(line)
out.close(); print(f'{n_g} genomes, {n_p} target proteins -> /tmp/db.faa')

## 4. Build DIAMOND database

In [ ]:
!./diamond makedb --in /tmp/db.faa -d /tmp/db --threads 4 2>&1 | tail -3
!ls -lh /tmp/db.dmnd

## 5. Build the query FASTA from our labeled organisms
Tag each query header `org|protein_id`; keep a protein_id -> (org, locus_tag) map from the GFFs so hits join back to labels.

In [ ]:
import glob, re, csv
# org accession -> organism name (manifest)
acc2org={}
for r in csv.DictReader(open('data/drive_import/labels/genome_cache_manifest.csv')):
    if r.get('accession'): acc2org[r['accession']]=r['organism']
# protein_id -> (org, locus_tag) from each labeled GFF
pid2gene={}
for gff in glob.glob('data/drive_import/genome_cache/GCF_*/genomic.gff'):
    acc=gff.split('/')[-2]; org=acc2org.get(acc)
    if not org: continue
    for line in open(gff):
        if '\tCDS\t' not in line: continue
        m=re.search(r'locus_tag=([^;\n]+)',line); p=re.search(r'protein_id=([^;\n]+)',line)
        if m and p: pid2gene[p.group(1)]=(org, m.group(1))
# query fasta = labeled proteins whose protein_id maps to a labeled gene
q=open('/tmp/query.faa','w'); nq=0
for faa in glob.glob('data/drive_import/genome_cache/GCF_*/protein.faa'):
    acc=faa.split('/')[-2]; org=acc2org.get(acc)
    if not org: continue
    keep=False
    for line in open(faa):
        if line.startswith('>'):
            pid=line[1:].split()[0]; keep=pid in pid2gene
            if keep: q.write('>'+pid+'\n'); nq+=1
        elif keep: q.write(line)
q.close(); print(f'{nq} query proteins (labeled genes) -> /tmp/query.faa')
import pickle; pickle.dump(pid2gene, open('/tmp/pid2gene.pkl','wb'))

## 6. DIAMOND search (the deep ortholog scan)

In [ ]:
# sensitive blastp; cap targets; require 30% id + 50% query coverage
!./diamond blastp -q /tmp/query.faa -d /tmp/db -o /tmp/hits.tsv \
    --outfmt 6 qseqid sseqid pident --id 30 --query-cover 50 \
    --max-target-seqs 2000 --more-sensitive --threads 4 2>&1 | tail -4
!wc -l /tmp/hits.tsv

## 7. Build deep features per labeled gene (retention breadth + mean identity)

In [ ]:
import pickle, collections, numpy as np
pid2gene=pickle.load(open('/tmp/pid2gene.pkl','rb'))
species_hit=collections.defaultdict(set); pid_sum=collections.defaultdict(float); pid_n=collections.defaultdict(int)
for line in open('/tmp/hits.tsv'):
    q,s,pid=line.rstrip('\n').split('\t')
    genome=s.split('|')[0]
    species_hit[q].add(genome); pid_sum[q]+=float(pid); pid_n[q]+=1
rows=[]
for pid,(org,lt) in pid2gene.items():
    breadth=len(species_hit.get(pid,()))
    mp=pid_sum[pid]/pid_n[pid] if pid_n[pid] else 0.0
    rows.append((org,lt,breadth,mp))
import pandas as pd
deep=pd.DataFrame(rows,columns=['organism','locus_tag','deep_retention','deep_pident'])
deep['deep_retention_frac']=deep.deep_retention/8388.0
deep.to_csv('outputs/orphan/deep_features.csv',index=False)
print(deep.describe())
print('genes with >=1 deep homolog:', (deep.deep_retention>0).mean())

## 8. Retrain with deep features and compare to the 48-organism baseline
Loads the prebuilt MSA cache, appends the two deep features to the focal vector, retrains leave-one-clade-out (pure NumPy, same model as `scripts/af_train.py`), and prints AUC with vs without deep features.

In [ ]:
import numpy as np, pandas as pd
Z=np.load('outputs/orphan/af_msa_cache.npz',allow_pickle=True)
foc=Z['foc']; msa=Z['msa']; mask=Z['msa_mask']; y=Z['y']; clade=Z['clade']
lt=Z['lt']; meta_org=Z['meta_org']; orgs=list(Z['orgs'])
deep=pd.read_csv('outputs/orphan/deep_features.csv')
dmap={(r.organism,r.locus_tag):(r.deep_retention_frac, r.deep_pident/100.0) for r in deep.itertuples()}
dv=np.array([dmap.get((orgs[meta_org[i]], str(lt[i])),(0.0,0.0)) for i in range(len(y))],np.float32)
print('deep coverage of cache genes:', (dv[:,0]>0).mean())
foc_deep=np.concatenate([foc,dv],1)

def fit(Xtr,ytr,Mtr,ktr,Xte,Mte,kte,it=400,lr=3e-3):
    # tiny attention-pool model (same shape as af_train), returns test probs
    rng=np.random.default_rng(0); Df=Xtr.shape[1]; Dm=Mtr.shape[2]; H=12
    Wq=rng.normal(0,.3,(Df,H)); Wk=rng.normal(0,.3,(Dm,H)); Wv=rng.normal(0,.3,(Dm,H))
    W1=rng.normal(0,.3,(Df+H,24)); b1=np.zeros(24); w2=rng.normal(0,.3,24); b2=0.0
    def fwd(X,M,k):
        q=X@Wq; K=M@Wk; V=M@Wv; s=np.einsum('bkh,bh->bk',K,q)/np.sqrt(H)
        s=np.where(k>0,s,-1e9); s=s-s.max(1,keepdims=True); e=np.exp(s)*(k>0); a=e/(e.sum(1,keepdims=True)+1e-9)
        ctx=np.einsum('bk,bkh->bh',a,V); z=np.concatenate([X,ctx],1); h=np.maximum(z@W1+b1,0)
        return 1/(1+np.exp(-(h@w2+b2))),(a,V,K,q,z,h)
    pos=ytr.mean() or 1e-6; m={};v={}
    for nm in ['Wq','Wk','Wv','W1','b1','w2']: m[nm]=0;v[nm]=0
    t=0
    for ep in range(it):
        bi=rng.permutation(len(Xtr))[:4096]
        p,(a,V,K,q,z,h)=fwd(Xtr[bi],Mtr[bi],ktr[bi]); yb=ytr[bi]
        cw=np.where(yb==1,0.5/pos,0.5/(1-pos)); dl=(p-yb)*cw/len(bi)
        gw2=h.T@dl; gb2=dl.sum(); dh=np.outer(dl,w2)*(h>0)
        gW1=z.T@dh; gb1=dh.sum(0); dz=dh@W1.T; dX=dz[:,:Xtr.shape[1]]; dctx=dz[:,Xtr.shape[1]:]
        da=np.einsum('bh,bkh->bk',dctx,V); dV=np.einsum('bk,bh->bkh',a,dctx)
        ds=(a*(da-(a*da).sum(1,keepdims=True)))/np.sqrt(H); dK=np.einsum('bk,bh->bkh',ds,q); dq=np.einsum('bk,bkh->bh',ds,K)
        gWq=Xtr[bi].T@dq; gWk=np.einsum('bkd,bkh->dh',Mtr[bi],dK); gWv=np.einsum('bkd,bkh->dh',Mtr[bi],dV)
        t+=1
        for nm,g,arr in [('Wq',gWq,Wq),('Wk',gWk,Wk),('Wv',gWv,Wv),('W1',gW1,W1),('b1',gb1,b1),('w2',gw2,w2)]:
            m[nm]=0.9*m[nm]+0.1*g; v[nm]=0.999*v[nm]+0.001*(g*g); arr-=lr*(m[nm]/(1-0.9**t))/(np.sqrt(v[nm]/(1-0.999**t))+1e-8)
        b2-=lr*gb2
    p,_=fwd(Xte,Mte,kte); return p
def auc(s,yy):
    yy=np.asarray(yy); n1=yy.sum(); n0=len(yy)-n1
    if n1==0 or n0==0: return float('nan')
    r=np.argsort(np.argsort(s))+1; return float((r[yy==1].sum()-n1*(n1+1)/2)/(n1*n0))

for label,F in [('baseline (48-org)',foc),('+ deep (8388-sp)',foc_deep)]:
    aucs=[]
    for cl in ['pseudomonas','ralstonia','shewanella','dickeya','burkholderia']:
        te=clade==cl; tr=~te
        if te.sum()<100: continue
        p=fit(F[tr],y[tr],msa[tr],mask[tr],F[te],msa[te],mask[te])
        aucs.append(auc(p,y[te]))
    print(f'{label:<22} mean LOCO AUC = {np.nanmean(aucs):.3f}   per-clade {[round(a,3) for a in aucs]}')

## Phase 2 (optional, heavier): deep dN/dS + phyletic-profile coevolution
- Add `--include cds` to the download, codon-align each labeled gene to its best hit per species, run Nei-Gojobori across the tree -> genome-wide deep selection (much stronger than the same-length-sister dN/dS we have now).
- Build the full phyletic profile matrix (gene-family x 8,388 species presence), feed its top PCs or the raw co-occurrence as the OuterProductMean input -> the coevolution signal AF actually exploits. This is the lever most likely to move the cross-clade ceiling.
- Save `deep_features.csv` back to the repo (`git add/commit/push`) to make the deepened atlas reproducible.